In [1]:
#import sys
#from pathlib import Path
#
#repo_root = Path.cwd().resolve()
#if not (repo_root / "solarrpy").exists():
#    candidate = repo_root.parent
#    if (candidate / "solarrpy").exists():
#        repo_root = candidate
#
#if str(repo_root) not in sys.path:
#    sys.path.insert(0, str(repo_root))

In [2]:
%reload_ext autoreload
%autoreload 2

In [3]:
import calendar
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from solarrpy.seasonalClearsky import SeasonalClearsky, control_seasonalClearsky, clearsky_outliers, clearsky_optimizer
from solarrpy.solarTransform import SolarTransform
from solarrpy.seasonalModel import SeasonalModel
from solarrpy.radiationModel import martingale_method_seasonal, reparam_seasonal_function, integral_sigma2_formula, integral_sigma_numeric
from solarrpy.gaussianMixture import GaussianMixtureModel, gm_moments

import warnings
warnings.filterwarnings('ignore')

/Users/rafaelvalente/Desktop/thesis/solarr/solarrpy/calibration.py:161: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
  OLS on CAMS clearsky regressed on ``(1, H_t, \sin, \cos)`` followed by


In [4]:
# Load your CAMS dataset for Bologna
# Expected columns: ['date', 'GHI']
df = pd.read_csv("../data/CAMS_data/CAMS_data_Bologna.csv")
#df = pd.read_csv("../data/Bologna.csv")
df['date'] = pd.to_datetime(df['date'])
df = df.drop(columns=['H0'])

# Bologna Coordinates used in the paper
LATITUDE = 44.4949
LONGITUDE = 11.3426
ALTITUDE = 71

# Define the training cutoffs based on Table A1 & A2
training_years = [2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]

In [5]:
plot_data = False
test_outputs = False

# Initialize storage for the resulting tables
table_a1_results = []
table_a2_results = []
table_a3_results = []

for year_i in range(2013, 2023):
    
    print(f"Processing training period: 2005-{year_i}...")
    
    # Isolate training data
    mask = df['date'].dt.year <= year_i
    df_train = df.loc[mask].copy()
    n_obs = len(df_train)

    print(f"Generating Table A1 for {year_i}...")
    
    # ---------------------------------------------------------
    # 1. Clear Sky Model (delta0, delta1, delta2, delta3)
    # ---------------------------------------------------------
        
    control_cs = control_seasonalClearsky(
        orders=1, order_H0=1, periods=365, 
        include_intercept=True, include_trend=False,
        delta0=1.4, lower=0, upper=3, by=0.001, ntol=0, quiet=False
    )

    clearsky_model = SeasonalClearsky(control=control_cs)

    clearsky_model.fit(
        x=df_train['GHI'], 
        date=df_train['date'], 
        lat=LATITUDE, 
        clearsky=df_train['clearsky'], # Initial target for OLS
        optimiser="delta_optimiser"
    )
    
    # The paper's formulation corresponds to: intercept, H0, sin, cos
    delta0, delta1, delta2, delta3 = clearsky_model._model.params.values[:4]
    delta0_err, delta1_err, delta2_err, delta3_err = clearsky_model._model.bse.values[:4]
    
    # Generate the bounded clear sky limit
    df_train['Ct'] = clearsky_model.predict(newdata=df_train)
    
    # MANDATORY SCRUBBING: Impute values where GHI > Ct or GHI < 0
    scrubbed = clearsky_outliers(df_train['GHI'].values, df_train['Ct'].values, df_train['date'])
    df_train['GHI_clean'] = scrubbed['x']

    # ==========================================
    # Plotting
    # ==========================================
    if plot_data:
        # Filter data between dates
        df_plot = df_train

        fig, axes = plt.subplots(1, 3, figsize=(15, 5))

        # Plot 1: CAMS vs GHI
        axes[0].plot(df_plot['n'], df_plot['clearsky'], linestyle=' ', marker='.', color='blue', label='CAMS')
        axes[0].plot(df_plot['n'], df_plot['GHI'], linestyle=' ', marker='.', color='black', label='GHI')
        axes[0].set_xlabel('Day of the year')
        axes[0].set_ylabel('Clear sky')
        axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, 1.15), ncol=2, frameon=False)
        axes[0].grid(True, linestyle='--', alpha=0.7)

        # Plot 2: Fitted vs CAMS
        axes[1].plot(df_plot['n'], df_plot['Ct'], linestyle=' ', marker='.', color='red', label='Fitted')
        axes[1].plot(df_plot['n'], df_plot['clearsky'], linestyle=' ', marker='.', color='blue', label='CAMS')
        axes[1].set_xlabel('Day of the year')
        axes[1].set_ylabel('Clear sky')
        axes[1].legend(loc='upper center', bbox_to_anchor=(0.5, 1.15), ncol=2, frameon=False)
        axes[1].grid(True, linestyle='--', alpha=0.7)

        # Plot 3: Fitted vs GHI
        axes[2].plot(df_plot['n'], df_plot['Ct'], linestyle=' ', marker='.', color='red', label='Fitted')
        axes[2].plot(df_plot['n'], df_plot['GHI'], linestyle=' ', marker='.', color='black', label='GHI')
        axes[2].set_xlabel('Day of the year')
        axes[2].set_ylabel('Clear sky')
        axes[2].legend(loc='upper center', bbox_to_anchor=(0.5, 1.15), ncol=2, frameon=False)
        axes[2].grid(True, linestyle='--', alpha=0.7)

        plt.tight_layout()
        plt.show()

    if test_outputs:
        # ==========================================
        # Test: imputed outliers
        # ==========================================
        # Impute outliers
        outliers = clearsky_outliers(df_train['clearsky'], df_train['Ct'], df_train['date'], quiet=True)
        df_train['clearsky'] = outliers['x']
        
        # Test tolerance parameter
        print("\033[1;35m---------------\033[0m \033[1;32m  Test clearskyModel_control and clearskyModel_fit \033[1;35m---------------\033[0m")
        
        passed_ntol = outliers['n'] <= control_cs['ntol']
        msg_ntol = "\033[1;32mPassed\033[0m!\n" if passed_ntol else "\033[1;31mNOT passed\033[0m! \n"
        
        print(f"Check if the number of outliers imputed is below {control_cs['ntol']}...({outliers['n']}) {msg_ntol}")
    
        # ==========================================
        # Test: delta parameter
        # ==========================================
        
        # Test delta parameter (Accessing mangled private attribute)
        delta = clearsky_model.delta    
        test_delta = (delta > control_cs['lower']) and (delta < control_cs['upper'])
        msg_delta = "\033[1;32mPassed\033[0m!\n" if test_delta else "\033[1;31mNOT passed\033[0m! \n"
        
        print(f"Check if the parameter delta is inside lower ({control_cs['lower']}) and upper ({control_cs['upper']})...({delta}) {msg_delta}")
        
        # ==========================================
        # Test: order of seasonal components
        # ==========================================
        
        # Count the number of parameters 
        n_params_target = 1 if control_cs['include_intercept'] else 0
        n_params_target += 1 if control_cs['include_trend'] else 0
        n_params_target += control_cs['orders'] * 2
        n_params_target += control_cs['order_H0']
        
        # Test if the number of parameters is correct 
        n_params = len(clearsky_model._model.params)
        
        # Print result 
        msg_params = "\033[1;32mPassed\033[0m!\n" if n_params == n_params_target else "\033[1;31mNOT passed\033[0m! \n"
        print(f"Check if the number of parameters is equal to {n_params_target}...({n_params}) {msg_params}")
        
        # ==========================================
        # Differential
        # ==========================================
        
        # Note differential when a trend is true is not implemented
        dt = 0.05
        n0 = 34

        num_diff = (clearsky_model.predict(n=n0 + dt) - clearsky_model.predict(n=n0)) / dt
        ana_diff = clearsky_model.differential(n=n0)
        
        print(f"Numerical Differential: \n{num_diff}")
        print(f"Analytical Differential: \n{ana_diff}")
        
    # ---------------------------------------------------------
    # 2. Bounded Transformation (alpha, beta)
    # ---------------------------------------------------------

    transform = SolarTransform(link='invgumbel')
    
    # Calculate preliminary risk driver using clean data
    Xt_raw = transform.X(df_train['GHI_clean'], df_train['Ct'])
    
    # Fit bounds
    bounds = transform.fit(Xt_raw, epsilon=0.001)
    alpha = bounds['alpha']
    beta = bounds['beta']
    
    # Execute actual transformation to Yt
    df_train['Yt'] = transform.RY(df_train['GHI_clean'], df_train['Ct'])
        
    # ---------------------------------------------------------
    # 3. Seasonal Mean of Yt (a_0, a_1, a_2)
    # ---------------------------------------------------------

    seasonal_Yt = SeasonalModel(orders=[1], periods=[365])
    
    seasonal_Yt.fit(data=df_train, target_col='Yt', time_col='n', include_intercept=True)
    df_train['Yt_bar'] = seasonal_Yt.predict(data=df_train, time_col='n')
    
    a0, a1, a2 = seasonal_Yt._model.params.values[:3]
    a0_err, a1_err, a2_err = seasonal_Yt._model.bse.values[:3]
    
    # Append to Table A1 storage
    table_a1_results.append({
        'Train years': f'2005-{year_i}', 'Obs.': n_obs,
        'alpha': alpha, 'beta': beta,
        'delta0': delta0, 'delta1': delta1, 'delta2': delta2, 'delta3': delta3,
        'delta0_err': delta0_err, 'delta1_err': delta1_err, 'delta2_err': delta2_err, 'delta3_err': delta3_err,
        'a0': a0, 'a1': a1, 'a2': a2,
        'a0_err': a0_err, 'a1_err': a1_err, 'a2_err': a2_err
    })

    print(f"Generating Table A2 for {year_i}...")

    # ---------------------------------------------------------
    # 4. Martingale Estimation (theta)
    # ---------------------------------------------------------
    
    theta = martingale_method_seasonal(df_train['Yt'].values, df_train['Yt_bar'].values)
    
    # ---------------------------------------------------------
    # 5. Seasonal Variance mapping (b, gamma, c)
    # ---------------------------------------------------------

    # Calculate dYt2 for the b parameters
    df_train['dYt2'] = np.nan
    df_train.loc[df_train.index[2:], 'dYt2'] = (df_train['Yt'].values[1:-1] - df_train['Yt'].values[:-2])**2
    
    fit_df = df_train.dropna(subset=['dYt2']).copy()
    
    seasonal_var = SeasonalModel(orders=[1], periods=[365])
    seasonal_var.fit(data=fit_df, target_col='dYt2', time_col='n', include_intercept=True)
    
    # Save the model coefficients
    b0, b1, b2 = seasonal_var._model.params.values[:3]
    b0_err, b1_err, b2_err = seasonal_var._model.bse.values[:3]
    
    # Reparametrize into continuous time
    reparam = reparam_seasonal_function([b0, b1, b2], theta)
    g0, g1, g2 = reparam['gamma']
    c0, c1, c2 = reparam['c_']
    
    # Append to Table A2 storage
    table_a2_results.append({
        'Year': year_i, 'theta': theta,
        'b0': b0, 'b1': b1, 'b2': b2,
        'b0_err': b0_err, 'b1_err': b1_err, 'b2_err': b2_err,
        'gamma0': g0, 'gamma1': g1, 'gamma2': g2,
        'c0': c0, 'c1': c1, 'c2': c2
    })
    '''
    print(f"Generating Table A3 for {year_i}...")

    # ---------------------------------------------------------
    # 6. Standardize the residuals
    # ---------------------------------------------------------
    
    # First, deseasonalize the transformed variable: Yt_tilde = Yt - Yt_bar
    Yt_tilde = df_train['Yt'] - df_train['Yt_bar']

    # Then apply the exact OU discretization formula
    df_train['eps'] = Yt_tilde - Yt_tilde.shift(1) * np.exp(-theta)

    # Setup the Integral Functions
    calc_J = integral_sigma2_formula(theta, reparam['gamma'])
    calc_I = integral_sigma_numeric(theta, reparam['c_'])

    # ---------------------------------------------------------
    # 9. Standardize the residuals
    # ---------------------------------------------------------
    
    # Compute J specifically for the actual days existing in our training set
    actual_days = df_train['n'].values

    # Evaluate J(t-1, t, t) and I(t-1, t, t) for the dataset
    J_cycle = calc_J(actual_days - 1, actual_days, actual_days)
    I_cycle = calc_I(actual_days - 1, actual_days, actual_days)

    # Standardize residuals
    df_train['eps_tilde'] = df_train['eps'] / np.sqrt(np.maximum(J_cycle, 1e-10)) # Avoid division by zero

    # Compute the continuous-time correction factor (I / sqrt(J))
    df_train['corr_factor'] = I_cycle / np.sqrt(J_cycle)

    results_month = []

    # ---------------------------------------------------------
    # 2. Iterate through months and fit the GMM
    # ---------------------------------------------------------
    for month in range(1, 13):
        mask = df_train['Month'] == month
        
        # Extract residuals and correction factors, dropping NaNs to keep arrays aligned
        valid_idx = df_train.loc[mask, 'eps_tilde'].dropna().index
        month_data = df_train.loc[valid_idx, 'eps_tilde'].values
        month_corr = df_train.loc[valid_idx, 'corr_factor'].mean()
        
        n_obs = len(month_data)
        if n_obs < 2:
            continue
            
        # Fit the Gaussian Mixture Model
        gmm = GaussianMixtureModel(components=2, maxit=5000)
        try:
            gmm.fit(month_data)
        except Exception as e:
            print(f"Fit failed for month {month}: {e}")
            continue
            
        # Extract uncorrected parameters
        mu_raw = gmm.means
        sd = gmm.sd
        p = gmm.p
        
        # Extract hard classification counts 
        # classify() returns 1-based indices: 1 = Cloudy (lower mean), 2 = Sunny (higher mean)
        classifications = gmm.fitted['classification']
        n1_cloudy = (classifications == 1).sum()
        n0_sunny = (classifications == 2).sum()
        
        # Apply the continuous-time discrepancy correction to the means
        mu_corrected = mu_raw * month_corr
        
        # Recompute theoretical moments using the corrected means to match Table A3
        moms = gm_moments(mu_corrected, sd, p).iloc[0]
        
        results_month.append([
            calendar.month_abbr[month],
            n_obs,
            p[0], mu_corrected[0], sd[0], n1_cloudy,
            mu_corrected[1], sd[1], n0_sunny,
            moms['mean'], moms['variance'], moms['skewness'], moms['kurtosis']
        ])
    
    table_a3_results.append(results_month)
    '''

Processing training period: 2005-2013...
Generating Table A1 for 2013...
No outliers!
Generating Table A2 for 2013...
Processing training period: 2005-2014...
Generating Table A1 for 2014...
No outliers!
Generating Table A2 for 2014...
Processing training period: 2005-2015...
Generating Table A1 for 2015...
No outliers!
Generating Table A2 for 2015...
Processing training period: 2005-2016...
Generating Table A1 for 2016...
No outliers!
Generating Table A2 for 2016...
Processing training period: 2005-2017...
Generating Table A1 for 2017...
No outliers!
Generating Table A2 for 2017...
Processing training period: 2005-2018...
Generating Table A1 for 2018...
No outliers!
Generating Table A2 for 2018...
Processing training period: 2005-2019...
Generating Table A1 for 2019...
No outliers!
Generating Table A2 for 2019...
Processing training period: 2005-2020...
Generating Table A1 for 2020...
No outliers!
Generating Table A2 for 2020...
Processing training period: 2005-2021...
Generating Tabl

## Constrained

In [6]:
"""
# ---------------------------------------------------------
# 1. Clear Sky Model (delta0, delta1, delta2, delta3)
# ---------------------------------------------------------

print("1. [START] Clear Sky Model (delta0, delta1, delta2, delta3)")

control_cs = control_seasonalClearsky(
    orders=1, order_H0=1, periods=365, 
    include_intercept=True, include_trend=False,
    delta0=1.4, lower=0, upper=3, by=0.001, ntol=0, quiet=False
)

# --- Stage 1: Unconstrained OLS fit (same as before) ---
clearsky_model = SeasonalClearsky(control=control_cs)
clearsky_model.fit(
    x=df_train['GHI'], 
    date=df_train['date'], 
    lat=LATITUDE, 
    clearsky=df_train['clearsky'],
    alt=ALTITUDE
)

# --- Stage 2: Constrained RLS fit (replaces delta scaling) ---
# clearsky_optimizer minimizes MSE subject to Ct >= GHI for all observations
clearsky_model = clearsky_optimizer(clearsky_model, data=df_train, ntol=0)

# Extract coefficients from the constrained model
delta0, delta1, delta2, delta3 = clearsky_model._model.params.values[:4]
delta0_err, delta1_err, delta2_err, delta3_err = clearsky_model._model.bse.values[:4]

# Generate the bounded clear sky limit
df_train['Ct'] = clearsky_model.predict(newdata=df_train)

# Verify the constraint is satisfied
n_violations = (df_train['GHI'] > df_train['Ct']).sum()
print(f"   Constraint violations after RLS: {n_violations}")

# MANDATORY SCRUBBING: Impute any residual outliers
scrubbed = clearsky_outliers(df_train['GHI'].values, df_train['Ct'].values, df_train['date'])
df_train['GHI_clean'] = scrubbed['x']

print("1. [END] Clear Sky Model (delta0, delta1, delta2, delta3)")
"""

'\n# ---------------------------------------------------------\n# 1. Clear Sky Model (delta0, delta1, delta2, delta3)\n# ---------------------------------------------------------\n\nprint("1. [START] Clear Sky Model (delta0, delta1, delta2, delta3)")\n\ncontrol_cs = control_seasonalClearsky(\n    orders=1, order_H0=1, periods=365, \n    include_intercept=True, include_trend=False,\n    delta0=1.4, lower=0, upper=3, by=0.001, ntol=0, quiet=False\n)\n\n# --- Stage 1: Unconstrained OLS fit (same as before) ---\nclearsky_model = SeasonalClearsky(control=control_cs)\nclearsky_model.fit(\n    x=df_train[\'GHI\'], \n    date=df_train[\'date\'], \n    lat=LATITUDE, \n    clearsky=df_train[\'clearsky\'],\n    alt=ALTITUDE\n)\n\n# --- Stage 2: Constrained RLS fit (replaces delta scaling) ---\n# clearsky_optimizer minimizes MSE subject to Ct >= GHI for all observations\nclearsky_model = clearsky_optimizer(clearsky_model, data=df_train, ntol=0)\n\n# Extract coefficients from the constrained model\

In [7]:
# Format Table A2
df_A1 = pd.DataFrame(table_a1_results)
df_A1 = df_A1.round({'alpha': 6, 'beta': 3, 'delta0': 2, 'delta1': 3, 'delta2': 4, 'delta3': 3, 'a0': 4, 'a1': 4, 'a2': 3})

# 1. Format the 'Train years' column
if 'Train years' not in df_A1.columns:
    df_A1['Train years'] = '2005-' + df_A1['Year'].astype(str)

# 2. Add 'Obs.' (using the values from your screenshot)
df_A1['Obs.'] = [3287, 3652, 4017, 4383, 4748, 5113, 5478, 5844, 6209, 6574]

# 3. Format the combined columns (Value + Standard Error with HTML <br>)
df_A1['a_0_fmt'] = df_A1.apply(lambda row: f"{row['a0']:.4f}<br>({row['a0_err']:.4f})", axis=1)
df_A1['a_1_fmt'] = df_A1.apply(lambda row: f"{row['a1']:.4f}<br>({row['a1_err']:.4f})", axis=1)
df_A1['a_2_fmt'] = df_A1.apply(lambda row: f"{row['a2']:.4f}<br>({row['a2_err']:.4f})", axis=1)

# 3. Format the combined columns (Value + Standard Error with HTML <br>)
df_A1['delta_0_fmt'] = df_A1.apply(lambda row: f"{row['delta0']:.4f}<br>({row['delta0_err']:.4f})", axis=1)
df_A1['delta_1_fmt'] = df_A1.apply(lambda row: f"{row['delta1']:.4f}<br>({row['delta1_err']:.4f})", axis=1)
df_A1['delta_2_fmt'] = df_A1.apply(lambda row: f"{row['delta2']:.4f}<br>({row['delta2_err']:.4f})", axis=1)
df_A1['delta_3_fmt'] = df_A1.apply(lambda row: f"{row['delta3']:.4f}<br>({row['delta3_err']:.4f})", axis=1)

# Format the remaining columns to standard decimal lengths
df_A1['alpha_fmt'] = df_A1['alpha'].apply(lambda x: f"{x:.6f}")
df_A1['beta_fmt'] = df_A1['beta'].apply(lambda x: f"{x:.3f}")

# 4. Select and rename columns for the final display
display_df = df_A1[['Train years', 'Obs.', 'alpha_fmt', 'beta_fmt', 
                    'delta_0_fmt', 'delta_1_fmt', 'delta_2_fmt', 'delta_3_fmt', 
                    'a_0_fmt', 'a_1_fmt', 'a_2_fmt']]

display_df.columns = ['Train years', 'N', 'alpha', 'beta', 
                    'delta0', 'delta1', 'delta2', 'delta3', 
                    'a0', 'a1', 'a2']

# Apply CSS Styling to mimic the LaTeX academic look
styles = [
    # Top and bottom thick horizontal rules
    {'selector': 'thead th', 'props': 'border-bottom: 1px solid black; border-top: 2px solid black; text-align: center; padding: 8px;'},
    {'selector': 'tbody tr:last-child td', 'props': 'border-bottom: 2px solid black;'},
    
    # Center all text and add vertical padding
    {'selector': 'tbody td', 'props': 'text-align: center; vertical-align: middle; padding: 8px;'},
    
    # Left-align the first column (Train years)
    {'selector': 'tbody td:nth-child(1)', 'props': 'text-align: left;'},
    
    # Selective vertical lines (matching the screenshot)
    # Column 3 is alpha, Column 5 is delta_0, Column 9 is a_0
    {'selector': 'th:nth-child(3), td:nth-child(3)', 'props': 'border-left: 1px solid black;'},
    {'selector': 'th:nth-child(5), td:nth-child(5)', 'props': 'border-left: 1px solid black;'},
    {'selector': 'th:nth-child(9), td:nth-child(9)', 'props': 'border-left: 1px solid black;'}
]

# Hide the default index and render
styled_table = display_df.style.set_table_styles(styles).hide(axis='index')
display(styled_table)

Train years,N,alpha,beta,delta0,delta1,delta2,delta3,a0,a1,a2
2005-2013,3287,0.000606,0.922,-1.1700(0.3598),0.9460(0.0483),0.0111(0.0358),0.5090(0.2058),-0.0762(0.0168),-0.0774(0.0238),-0.3920(0.0238)
2005-2014,3652,0.000159,0.923,-1.0800(0.3409),0.9360(0.0457),0.0250(0.0339),0.4510(0.1950),-0.0922(0.0159),-0.0720(0.0225),-0.3870(0.0225)
2005-2015,4017,0.000958,0.922,-1.3100(0.3224),0.9670(0.0432),0.0004(0.0321),0.5860(0.1844),-0.0818(0.0150),-0.0639(0.0212),-0.3750(0.0212)
2005-2016,4383,0.000375,0.922,-1.2400(0.3104),0.9570(0.0416),0.0061(0.0309),0.5410(0.1776),-0.0833(0.0144),-0.0695(0.0204),-0.3730(0.0204)
2005-2017,4748,0.000250,0.923,-1.1600(0.2972),0.9460(0.0399),0.0190(0.0296),0.4960(0.1700),-0.0663(0.0138),-0.0671(0.0195),-0.3690(0.0195)
2005-2018,5113,0.000363,0.922,-1.1300(0.2873),0.9410(0.0385),0.0223(0.0286),0.4840(0.1643),-0.0662(0.0133),-0.0795(0.0188),-0.3760(0.0188)
2005-2019,5478,0.000168,0.923,-1.3000(0.2776),0.9630(0.0372),0.0125(0.0276),0.5880(0.1588),-0.0581(0.0128),-0.0761(0.0181),-0.3620(0.0181)
2005-2020,5844,0.001180,0.922,-1.3200(0.2679),0.9670(0.0359),0.0115(0.0267),0.6020(0.1533),-0.0464(0.0123),-0.0666(0.0174),-0.3540(0.0174)
2005-2021,6209,0.000910,0.922,-1.5000(0.2610),0.9920(0.0350),-0.0009(0.0260),0.7010(0.1493),-0.0475(0.0118),-0.0617(0.0167),-0.3520(0.0167)
2005-2022,6574,0.000090,0.923,-1.6600(0.2548),1.0150(0.0342),-0.0169(0.0253),0.7960(0.1457),-0.0423(0.0115),-0.0566(0.0163),-0.3500(0.0163)


In [8]:
# Format Table A1
df_A2 = pd.DataFrame(table_a2_results)
df_A2 = df_A2.round({'theta': 3, 'b0': 3, 'b1': 3, 'b2': 3, 'gamma0': 3, 'gamma1': 3, 'gamma2': 3, 'c0': 2, 'c1': 3, 'c2': 3})

# 1. Format the 'Train years' column
if 'Train years' not in df_A2.columns:
    df_A2['Train years'] = '2005-' + df_A2['Year'].astype(str)

# 2. Add 'Obs.' (using the values from your screenshot)
df_A2['Obs.'] = [3287, 3652, 4017, 4383, 4748, 5113, 5478, 5844, 6209, 6574]

# 3. Format the combined columns (Value + Standard Error with HTML <br>)
df_A2['b_0_fmt'] = df_A2.apply(lambda row: f"{row['b0']:.4f}<br>({row['b0_err']:.4f})", axis=1)
df_A2['b_1_fmt'] = df_A2.apply(lambda row: f"{row['b1']:.4f}<br>({row['b1_err']:.4f})", axis=1)
df_A2['b_2_fmt'] = df_A2.apply(lambda row: f"{row['b2']:.4f}<br>({row['b2_err']:.4f})", axis=1)

# Format the remaining columns to standard decimal lengths
df_A2['gamma_0_fmt'] = df_A2['gamma0'].apply(lambda x: f"{x:.6f}")
df_A2['gamma_1_fmt'] = df_A2['gamma1'].apply(lambda x: f"{x:.6f}")
df_A2['gamma_2_fmt'] = df_A2['gamma2'].apply(lambda x: f"{x:.6f}")

df_A2['c_0_fmt'] = df_A2['c0'].apply(lambda x: f"{x:.6f}")
df_A2['c_1_fmt'] = df_A2['c1'].apply(lambda x: f"{x:.6f}")
df_A2['c_2_fmt'] = df_A2['c2'].apply(lambda x: f"{x:.6f}")

df_A2['theta_fmt'] = df_A2['theta'].apply(lambda x: f"{x:.6f}")

# 4. Select and rename columns for the final display
display_df = df_A2[['Train years', 'Obs.', 'theta_fmt',
                    'b_0_fmt', 'b_1_fmt', 'b_2_fmt', 
                    'gamma_0_fmt', 'gamma_1_fmt', 'gamma_2_fmt', 
                    'c_0_fmt', 'c_1_fmt', 'c_2_fmt']]

display_df.columns = ['Train years', 'N', 'theta',
                    'b0', 'b1', 'b2', 
                    'gamma0', 'gamma1', 'gamma2', 
                    'c0', 'c1', 'c2']

# Apply CSS Styling to mimic the LaTeX academic look
styles = [
    # Top and bottom thick horizontal rules
    {'selector': 'thead th', 'props': 'border-bottom: 1px solid black; border-top: 2px solid black; text-align: center; padding: 8px;'},
    {'selector': 'tbody tr:last-child td', 'props': 'border-bottom: 2px solid black;'},
    
    # Center all text and add vertical padding
    {'selector': 'tbody td', 'props': 'text-align: center; vertical-align: middle; padding: 8px;'},
    
    # Left-align the first column (Train years)
    {'selector': 'tbody td:nth-child(1)', 'props': 'text-align: left;'},
    
    # Selective vertical lines (matching the screenshot)
    # Column 3 is alpha, Column 5 is delta_0, Column 9 is a_0
    {'selector': 'th:nth-child(3), td:nth-child(3)', 'props': 'border-left: 1px solid black;'},
    {'selector': 'th:nth-child(4), td:nth-child(4)', 'props': 'border-left: 1px solid black;'},
    {'selector': 'th:nth-child(7), td:nth-child(7)', 'props': 'border-left: 1px solid black;'},
    {'selector': 'th:nth-child(10), td:nth-child(10)', 'props': 'border-left: 1px solid black;'},
    {'selector': 'th:nth-child(13), td:nth-child(13)', 'props': 'border-left: 1px solid black;'}
]

# Hide the default index and render
styled_table = display_df.style.set_table_styles(styles).hide(axis='index')
display(styled_table)

Train years,N,theta,b0,b1,b2,gamma0,gamma1,gamma2,c0,c1,c2
2005-2013,3287,0.961000,1.1610(0.0970),0.3840(0.1372),0.7240(0.1372),1.360000,0.452000,0.847000,2.610000,0.855000,1.635000
2005-2014,3652,0.982000,1.1670(0.1024),0.3970(0.1448),0.7140(0.1448),1.358000,0.464000,0.830000,2.670000,0.897000,1.637000
2005-2015,4017,0.969000,1.1250(0.0760),0.3190(0.1075),0.6650(0.1075),1.314000,0.375000,0.776000,2.550000,0.712000,1.509000
2005-2016,4383,0.985000,1.1470(0.0784),0.3320(0.1109),0.6880(0.1109),1.333000,0.388000,0.799000,2.630000,0.751000,1.581000
2005-2017,4748,0.993000,1.1410(0.0765),0.3380(0.1082),0.7040(0.1082),1.323000,0.394000,0.815000,2.630000,0.769000,1.625000
2005-2018,5113,1.006000,1.1440(0.0692),0.3320(0.0979),0.7160(0.0979),1.320000,0.385000,0.825000,2.660000,0.760000,1.667000
2005-2019,5478,1.005000,1.1400(0.0705),0.3180(0.0996),0.6820(0.0996),1.316000,0.369000,0.786000,2.640000,0.728000,1.586000
2005-2020,5844,0.995000,1.1030(0.0538),0.2560(0.0762),0.6330(0.0761),1.278000,0.298000,0.733000,2.540000,0.581000,1.464000
2005-2021,6209,1.005000,1.0960(0.0523),0.2390(0.0740),0.6310(0.0740),1.265000,0.278000,0.728000,2.540000,0.545000,1.469000
2005-2022,6574,1.003000,1.0970(0.0631),0.2580(0.0893),0.6340(0.0893),1.268000,0.300000,0.732000,2.540000,0.590000,1.474000


In [9]:
# ---------------------------------------------------------
# 3. Build and Format the MultiIndex DataFrame
# ---------------------------------------------------------
columns = pd.MultiIndex.from_tuples([
    ('', 'Month'),
    ('', 'N'),
    ('Cloudy $B_t = 1$', '$\\mathbb{P}(B_t=1)$'),
    ('Cloudy $B_t = 1$', '$\\mu_1$'),
    ('Cloudy $B_t = 1$', '$\\sigma_1$'),
    ('Cloudy $B_t = 1$', '$N_1$'),
    ('Sunny $B_t = 0$', '$\\mu_0$'),
    ('Sunny $B_t = 0$', '$\\sigma_0$'),
    ('Sunny $B_t = 0$', '$N_0$'),
    ('Gaussian mixture moments', 'Mean'),
    ('Gaussian mixture moments', 'Variance'),
    ('Gaussian mixture moments', 'Skewness'),
    ('Gaussian mixture moments', 'Kurtosis')
])

df_A3 = pd.DataFrame(table_a3_results[-1], columns=columns)

# Apply specific float formatting based on the screenshot
format_dict = {
    ('Cloudy $B_t = 1$', '$\\mathbb{P}(B_t=1)$'): '{:.2f}',
    ('Cloudy $B_t = 1$', '$\\mu_1$'): '{:.2f}',
    ('Cloudy $B_t = 1$', '$\\sigma_1$'): '{:.2f}',
    ('Sunny $B_t = 0$', '$\\mu_0$'): '{:.2f}',
    ('Sunny $B_t = 0$', '$\\sigma_0$'): '{:.2f}',
    ('Gaussian mixture moments', 'Mean'): '{:.3f}',
    ('Gaussian mixture moments', 'Variance'): '{:.2f}',
    ('Gaussian mixture moments', 'Skewness'): '{:.2f}',
    ('Gaussian mixture moments', 'Kurtosis'): '{:.3f}'
}

for col, fmt in format_dict.items():
    df_A3[col] = df_A3[col].apply(lambda x: fmt.format(x))

# ---------------------------------------------------------
# 4. Apply Academic CSS Styling
# ---------------------------------------------------------
styles = [
    # Multi-level header lines
    {'selector': 'thead th', 'props': 'border-bottom: 1px solid black; text-align: center; padding: 6px;'},
    {'selector': 'thead tr:first-child th', 'props': 'border-top: 2px solid black; border-bottom: 1px solid black;'},
    {'selector': 'tbody tr:last-child td', 'props': 'border-bottom: 2px solid black;'},
    
    # Text alignment
    {'selector': 'tbody td', 'props': 'text-align: center; vertical-align: middle; padding: 6px;'},
    {'selector': 'tbody td:nth-child(1)', 'props': 'text-align: left;'},
    
    # Vertical section dividers (Columns 3, 7, and 10 in 1-based CSS)
    {'selector': 'th:nth-child(3), td:nth-child(3)', 'props': 'border-left: 1px solid black;'},
    {'selector': 'th:nth-child(7), td:nth-child(7)', 'props': 'border-left: 1px solid black;'},
    {'selector': 'th:nth-child(10), td:nth-child(10)', 'props': 'border-left: 1px solid black;'}
]

styled_table_A3 = df_A3.style.set_table_styles(styles).hide(axis='index')
display(styled_table_A3)

IndexError: list index out of range

### Load Files

In [ ]:
# Load coefficients
#df_coefficients = pd.read_parquet('../results/A1_deltas.parquet', engine='pyarrow')

# Load the dictionary of arrays
#with np.load('../results/A1_Ct_arrays.npz') as data:
#    Ct_arrays = {int(year): data[year] for year in data.files}